In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Find project root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
features_path = PROCESSED / "master_panel_features.parquet"

print("Features file exists?", features_path.exists())

df = pd.read_parquet(features_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Features file exists? True
Shape: (479783, 41)

Columns:
['permno', 'permco', 'ticker', 'cusip', 'issuernm', 'siccd', 'naics', 'dlyclose', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyprc', 'dlyret', 'dlyretx', 'dlyvol', 'shrout', 'dlycap', 'sprtrn', 'vwretd', 'ewretd', 'date', 'quarter', 'numeric_transparency', 'analyst_selectivity_ratio', 'language_complexity', 'net_positivity', 'quarter_end', 'ret_5d', 'ret_20d', 'ret_60d', 'vol_20d', 'vol_60d', 'avg_volume_20d', 'relative_volume', 'positivity_delta', 'transparency_delta', 'complexity_delta', 'selectivity_delta', 'credibility_score', 'risk_score', 'misalignment_score']


,permno,permco,ticker,cusip,issuernm,siccd,naics,dlyclose,dlyopen,dlyhigh,...,vol_60d,avg_volume_20d,relative_volume,positivity_delta,transparency_delta,complexity_delta,selectivity_delta,credibility_score,risk_score,misalignment_score
0,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.21,57.10,57.100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
1,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.92,56.39,57.345,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.64,57.40,57.700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
3,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,57.45,56.95,57.630,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
4,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,58.39,57.33,58.540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN


In [3]:
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.dropna(subset=["ticker", "date"]).copy()
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

print(df.shape)
df[["ticker", "date"]].head()

(479783, 41)


,ticker,date
0,A,2014-01-02
1,A,2014-01-03
2,A,2014-01-06
3,A,2014-01-07
4,A,2014-01-08


In [4]:
latest_snapshot = (
    df.sort_values(["ticker", "date"])
      .groupby("ticker", as_index=False)
      .tail(1)
      .copy()
)

print("Latest snapshot shape:", latest_snapshot.shape)

latest_snapshot[
    [
        "ticker",
        "date",
        "dlyclose",
        "ret_20d",
        "vol_20d",
        "credibility_score",
        "risk_score",
        "misalignment_score",
    ]
].head(10)

Latest snapshot shape: (180, 41)


,ticker,date,dlyclose,ret_20d,vol_20d,credibility_score,risk_score,misalignment_score
2767,A,2024-12-31,134.34,-0.037541,0.012083,-1.116,5.740853,1.137541
5535,AAPL,2024-12-31,250.42,0.045202,0.010385,-0.918,5.902784,1.414798
8303,ABBV,2024-12-31,177.70,-0.022391,0.012264,-1.052,6.413460,1.632391
11071,ABT,2024-12-31,113.11,-0.031675,0.007710,-1.086,5.758547,1.421675
13839,ADBE,2024-12-31,444.68,-0.138551,0.034389,-0.946,7.848664,2.908551
16607,ADI,2024-12-31,212.46,-0.047777,0.013245,-1.044,6.487133,1.187777
19375,ADM,2024-12-31,50.52,-0.069099,0.014137,-1.516,6.248687,0.889099
22143,ADP,2024-12-31,292.73,-0.043460,0.010057,-0.898,5.526786,2.113460
24911,ADSK,2024-12-31,295.57,-0.003641,0.012254,-1.294,6.629075,1.593641
27679,AEE,2024-12-31,89.14,-0.040267,0.008990,-1.092,7.259124,1.540267


In [5]:
latest_snapshot_path = PROCESSED / "latest_company_snapshot.parquet"
latest_snapshot.to_parquet(latest_snapshot_path, index=False)

print("Saved:", latest_snapshot_path)

Saved: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\latest_company_snapshot.parquet


In [6]:
high_risk = (
    latest_snapshot.sort_values("risk_score", ascending=False)
    .loc[:, [
        "ticker",
        "date",
        "dlyclose",
        "ret_5d",
        "ret_20d",
        "vol_20d",
        "relative_volume",
        "risk_score",
        "credibility_score",
        "misalignment_score",
    ]]
    .reset_index(drop=True)
)

high_risk.head(20)

,ticker,date,dlyclose,ret_5d,ret_20d,vol_20d,relative_volume,risk_score,credibility_score,misalignment_score
0,ADBE,2024-12-31,444.68,-0.004611,-0.138551,0.034389,0.439931,7.848664,-0.946,2.908551
1,ICE,2024-12-31,149.01,-0.007658,-0.058627,0.008330,0.514489,7.617643,-1.788,1.038627
2,NWSA,2024-12-31,27.54,-0.018182,-0.064856,0.008778,0.792798,7.526206,-1.138,1.194856
3,CI,2024-12-31,276.14,-0.019494,-0.180739,0.024492,0.479807,7.424862,-1.584,1.540739
4,CVS,2024-12-31,44.89,0.016991,-0.240183,0.026418,0.754280,7.341509,-1.374,1.860183
5,PRU,2024-12-31,118.53,-0.000253,-0.076941,0.013612,0.485015,7.292082,-1.512,1.966941
6,MCK,2024-12-31,569.91,-0.011877,-0.082699,0.008791,0.550793,7.280141,-1.552,1.372699
7,AEE,2024-12-31,89.14,-0.006132,-0.040267,0.008990,0.950844,7.259124,-1.092,1.540267
8,EMR,2024-12-31,123.93,0.000000,-0.073559,0.013443,0.719448,7.255222,-1.046,1.503559
9,GPC,2024-12-31,116.76,0.004992,-0.084451,0.011245,0.766308,7.248471,-1.566,0.774451


In [7]:
high_risk_path = PROCESSED / "high_risk_companies.parquet"
high_risk.to_parquet(high_risk_path, index=False)

print("Saved:", high_risk_path)

Saved: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\high_risk_companies.parquet


In [8]:
high_credibility = (
    latest_snapshot.sort_values("credibility_score", ascending=False)
    .loc[:, [
        "ticker",
        "date",
        "dlyclose",
        "ret_5d",
        "ret_20d",
        "net_positivity",
        "numeric_transparency",
        "language_complexity",
        "credibility_score",
        "risk_score",
        "misalignment_score",
    ]]
    .reset_index(drop=True)
)

high_credibility.head(20)

,ticker,date,dlyclose,ret_5d,ret_20d,net_positivity,numeric_transparency,language_complexity,credibility_score,risk_score,misalignment_score
0,AVGO,2024-12-31,231.84,-0.002195,0.392349,0.45,5.49,10.47,0.282,5.106573,0.057651
1,DOC,2024-12-31,20.27,0.004460,-0.063741,1.53,2.67,10.38,-0.396,5.235636,1.593741
2,PNC,2024-12-31,192.85,-0.001398,-0.087576,0.85,3.44,10.71,-0.426,5.408568,0.937576
3,ECL,2024-12-31,234.32,-0.019951,-0.056873,2.22,2.63,11.97,-0.454,6.024078,2.276873
4,ITW,2024-12-31,253.56,-0.015836,-0.089551,1.61,3.47,12.76,-0.520,6.431433,1.699551
5,VTR,2024-12-31,58.89,-0.000679,-0.054128,2.36,2.40,12.18,-0.532,6.127788,2.414128
6,FISV,2023-06-06,114.23,0.011243,-0.044100,1.44,3.82,13.22,-0.540,6.643315,1.484100
7,MSI,2024-12-31,462.23,-0.016050,-0.069398,1.45,2.93,11.52,-0.552,5.802863,1.519398
8,UPS,2024-12-31,126.10,0.002783,-0.059096,1.40,2.67,11.00,-0.572,5.543456,1.459096
9,AME,2024-12-31,180.26,-0.009941,-0.078237,1.66,2.44,11.07,-0.574,5.583821,1.738237


In [9]:
high_credibility_path = PROCESSED / "high_credibility_companies.parquet"
high_credibility.to_parquet(high_credibility_path, index=False)

print("Saved:", high_credibility_path)

Saved: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\high_credibility_companies.parquet


In [10]:
signal_screener = latest_snapshot[
    [
        "ticker",
        "date",
        "dlyclose",
        "ret_5d",
        "ret_20d",
        "ret_60d",
        "vol_20d",
        "vol_60d",
        "relative_volume",
        "net_positivity",
        "numeric_transparency",
        "language_complexity",
        "analyst_selectivity_ratio",
        "positivity_delta",
        "transparency_delta",
        "complexity_delta",
        "selectivity_delta",
        "credibility_score",
        "risk_score",
        "misalignment_score",
    ]
].copy()

signal_screener.head(10)

,ticker,date,dlyclose,ret_5d,ret_20d,ret_60d,vol_20d,vol_60d,relative_volume,net_positivity,numeric_transparency,language_complexity,analyst_selectivity_ratio,positivity_delta,transparency_delta,complexity_delta,selectivity_delta,credibility_score,risk_score,misalignment_score
2767,A,2024-12-31,134.34,-0.001932,-0.037541,-0.073326,0.012083,0.015691,0.466968,1.10,1.82,11.42,46.67,-0.14,-0.06,0.66,-12.15,-1.116,5.740853,1.137541
5535,AAPL,2024-12-31,250.42,-0.018999,0.045202,0.104145,0.010385,0.010795,0.847932,1.46,2.16,11.83,19.05,-0.11,0.17,-0.21,0.45,-0.918,5.902784,1.414798
8303,ABBV,2024-12-31,177.70,-0.003868,-0.022391,-0.085388,0.012264,0.021616,0.567721,1.61,2.15,12.78,28.57,0.31,0.12,0.02,-15.43,-1.052,6.413460,1.632391
11071,ABT,2024-12-31,113.11,-0.010498,-0.031675,0.004173,0.007710,0.010599,0.744383,1.39,1.63,11.47,31.58,-0.18,0.20,-0.40,3.01,-1.086,5.758547,1.421675
13839,ADBE,2024-12-31,444.68,-0.004611,-0.138551,-0.123300,0.034389,0.024594,0.439931,2.77,2.61,15.49,15.15,-0.13,1.08,0.23,-3.03,-0.946,7.848664,2.908551
16607,ADI,2024-12-31,212.46,-0.011308,-0.047777,-0.069097,0.013245,0.016952,0.556941,1.14,2.70,12.90,28.00,0.32,0.70,0.03,-1.63,-1.044,6.487133,1.187777
19375,ADM,2024-12-31,50.52,0.002182,-0.069099,-0.147054,0.014137,0.014894,0.660446,0.82,1.59,12.40,66.67,-0.86,-0.36,0.14,-4.76,-1.516,6.248687,0.889099
22143,ADP,2024-12-31,292.73,-0.004827,-0.043460,0.026547,0.010057,0.010525,0.503379,2.07,1.18,10.99,35.29,0.33,-0.91,-0.16,-3.60,-0.898,5.526786,2.113460
24911,ADSK,2024-12-31,295.57,-0.006454,-0.003641,0.090021,0.012254,0.017360,0.512420,1.59,1.79,13.23,34.62,-0.10,0.31,-0.20,8.69,-1.294,6.629075,1.593641
27679,AEE,2024-12-31,89.14,-0.006132,-0.040267,0.015956,0.008990,0.011599,0.950844,1.50,3.00,14.46,15.38,-0.17,0.56,1.09,-15.39,-1.092,7.259124,1.540267


In [11]:
signal_screener_path = PROCESSED / "signal_screener.parquet"
signal_screener.to_parquet(signal_screener_path, index=False)

print("Saved:", signal_screener_path)

Saved: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\signal_screener.parquet


In [12]:
print("Latest snapshot:", latest_snapshot.shape)
print("High risk:", high_risk.shape)
print("High credibility:", high_credibility.shape)
print("Signal screener:", signal_screener.shape)

print("\nDuplicate tickers in latest snapshot:",
      latest_snapshot["ticker"].duplicated().sum())

Latest snapshot: (180, 41)
High risk: (180, 10)
High credibility: (180, 11)
Signal screener: (180, 20)

Duplicate tickers in latest snapshot: 0


In [13]:
check_latest = pd.read_parquet(latest_snapshot_path)
check_risk = pd.read_parquet(high_risk_path)
check_cred = pd.read_parquet(high_credibility_path)
check_screen = pd.read_parquet(signal_screener_path)

print("Reloaded latest:", check_latest.shape)
print("Reloaded risk:", check_risk.shape)
print("Reloaded credibility:", check_cred.shape)
print("Reloaded screener:", check_screen.shape)

Reloaded latest: (180, 41)
Reloaded risk: (180, 10)
Reloaded credibility: (180, 11)
Reloaded screener: (180, 20)
